# 📚 توثيق قاعدة بيانات برامج دعم المرضى (PSP)
## RubikCare - نظام برامج دعم المرضى

---

## القسم الأول: الجداول الموجودة (Existing Tables) - 10 جداول

---

### جدول 1: `PSPPrograms`

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | ProgramID (int) |
| **الغرض** | تخزين برامج دعم المرضى التي تنشئها شركات الأدوية |

**الأعمدة الرئيسية:**
- ProgramCode (nvarchar(50), Unique) - رمز فريد للبرنامج
- ProgramNameAr/En (nvarchar(200)) - اسم البرنامج
- StartDate/EndDate (datetime) - مدة البرنامج
- PharmaCompanyID (int, FK→Organizations) - الشركة المالكة
- Status (int) - حالة البرنامج

**العلاقات:**
- واحد → متعدد مع PSPProgramMedications
- واحد → متعدد مع PSPParticipations
- واحد → متعدد مع PSPeRXs
- واحد → متعدد مع PSPDispensationPlans
- واحد → متعدد مع PSPProgramSpecialities

---

### جدول 2: `PSPProgramMedications`

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | ProgramMedicationID (int) |
| **الغرض** | ربط برامج الدعم بالأدوية المشمولة |

**الأعمدة الرئيسية:**
- ProgramID (int, FK→PSPPrograms)
- MedicationID (int, FK→Medications)
- CoveragePercentage (decimal(5,2)) - نسبة التغطية
- MaxQuantityPerDispensation (int) - أقصى كمية لكل صرف

**العلاقات:**
- متعدد → واحد مع PSPPrograms
- متعدد → واحد مع Medications
- واحد → متعدد مع PSPDispensationPlanMedications

---

### جدول 3: `PSPParticipations`

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | ParticipationID (int) |
| **الغرض** | تسجيل مشاركة المنظمات (عيادات، صيدليات) في البرامج |

**الأعمدة الرئيسية:**
- ProgramID (int, FK→PSPPrograms)
- ParticipantOrganizationID (int, FK→Organizations)
- Status (int) - حالة المشاركة
- JoinDate (datetime) - تاريخ الانضمام

**العلاقات:**
- واحد → متعدد مع PSPPatients
- متعدد → واحد مع PSPPrograms

**قيود:**
- Unique: (ProgramID, ParticipantOrganizationID)

---

### جدول 4: `PSPPatients`

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | PatientID (int) |
| **الغرض** | تسجيل مرضى برامج الدعم |

**الأعمدة الرئيسية:**
- ParticipationID (int, FK→PSPParticipations)
- PatientProfileID (int, FK→UserProfiles)
- Status (int) - حالة المريض
- EnrollmentDate (datetime) - تاريخ التسجيل

**العلاقات:**
- واحد → متعدد مع PSPeRXs
- متعدد → واحد مع PSPParticipations

**قيود:**
- Unique: (ParticipationID, PatientProfileID)

---

### جدول 5: `PSPeRXs`

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | ERxID (int) |
| **الغرض** | الوصفات الإلكترونية للمرضى |

**الأعمدة الرئيسية:**
- ERxCode (nvarchar(50), Unique) - رمز الوصفة
- PatientID (int, FK→PSPPatients)
- DoctorID (nvarchar(450), FK→AspNetUsers)
- ProgramID (int, FK→PSPPrograms)
- MedicationID (int, FK→Medications)
- Quantity (int) - الكمية الموصوفة

**العلاقات:**
- واحد → متعدد مع PSPDispensations

---

### جدول 6: `PSPDispensationPlans`

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | PlanID (int) |
| **الغرض** | خطط الصرف لبرامج الدعم |

**الأعمدة الرئيسية:**
- ProgramID (int, FK→PSPPrograms)
- PlanName (nvarchar(200)) - اسم الخطة
- IsActive (bit) - هل الخطة نشطة؟

**العلاقات:**
- واحد → متعدد مع PSPDispensations
- واحد → متعدد مع PSPDispensationPlanMedications

**قيود:**
- Unique: (ProgramID, PlanName)

---

### جدول 7: `PSPDispensations`

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | DispensationID (int) |
| **الغرض** | تسجيل عمليات الصرف الفعلية |

**الأعمدة الرئيسية:**
- DispensationCode (nvarchar(50), Unique)
- ERxID (int, FK→PSPeRXs)
- PlanID (int, FK→PSPDispensationPlans)
- PharmacyID (int, FK→Organizations)
- PharmacistID (nvarchar(450), FK→AspNetUsers)
- PSPCoverage (decimal(18,2)) - تغطية البرنامج
- PatientPayment (decimal(18,2)) - ما يدفعه المريض

**الدقة العشرية:** (18,2) لجميع الأعمدة المالية

---

### جدول 8: `PSPProgramSpecialities`

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | PSPProgramSpecialityID (int) |
| **الغرض** | ربط برامج الدعم بالتخصصات الطبية |

**الأعمدة الرئيسية:**
- ProgramID (int, FK→PSPPrograms)
- SpecialityID (int, FK→Specialities)

**قيود:**
- Unique: (ProgramID, SpecialityID)

---

### جدول 9: `PSPInvitations`

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | InvitationID (int) |
| **الغرض** | تسجيل دعوات المشاركة في البرامج |

**الأعمدة الرئيسية:**
- ProgramID (int, FK→PSPPrograms)
- InvitedOrganizationID (int, FK→Organizations)
- Status (int) - حالة الدعوة
- InvitationDate (datetime) - تاريخ الدعوة

**قيود:**
- Unique: (ProgramID, InvitedOrganizationID)

---

### جدول 10: `PSPDispensationPlanMedications`

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | PlanMedicationID (int) |
| **الغرض** | تفاصيل الأدوية في خطة الصرف |

**الأعمدة الرئيسية:**
- PlanID (int, FK→PSPDispensationPlans)
- ProgramMedicationID (int, FK→PSPProgramMedications)
- Quantity (int) - الكمية في كل صرف
- DiscountPercentage (decimal(18,2)) - نسبة الخصم

**الدقة العشرية:** (18,2) لـ DiscountPercentage

---

## القسم الثاني: الجداول الجديدة (New Tables) - 7 جداول

---

### جدول 11: `PSPRequiredTests` - التحاليل المطلوبة

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | RequiredTestID (int) |
| **الغرض** | تحديد التحاليل الطبية المطلوبة في برنامج الدعم |

**الأعمدة الرئيسية:**
- ProgramID (int, FK→PSPPrograms)
- TestName (nvarchar(200)) - اسم التحليل
- NormalRangeMin/Max (decimal(18,2)) - النطاق الطبيعي
- RequiredFrom (int) - 1=مريض, 2=طبيب, 3=كلاهما
- RequiredTiming (int) - 1=قبل, 2=أثناء, 3=بعد, 4=دوري
- FrequencyDays (int) - للتكرار الدوري
- IsMandatory (bit) - هل هو إلزامي؟

**العلاقات:**
- واحد → متعدد مع PSPTestResults

---

### جدول 12: `PSPRequiredFollowUps` - مواعيد المتابعة المطلوبة

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | RequiredFollowUpID (int) |
| **الغرض** | تحديد مواعيد المتابعة المطلوبة |

**الأعمدة الرئيسية:**
- ProgramID (int, FK→PSPPrograms)
- FollowUpTitle (nvarchar(200))
- FollowUpType (int) - 1=عيادية, 2=هاتفية, 3=تطبيق, 4=منزلية
- RequiredFrom (int) - 1=مريض, 2=طبيب, 3=كلاهما
- RequiredTiming (int) - 1=قبل, 2=أثناء, 3=بعد, 4=دوري
- FrequencyDays (int) - للتكرار الدوري
- ReminderDaysBefore (int) - تذكير قبل كذا يوم

**العلاقات:**
- واحد → متعدد مع PSPFollowUpRecords

---

### جدول 13: `PSPRequiredDataEntries` - البيانات المطلوب تسجيلها

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | RequiredDataEntryID (int) |
| **الغرض** | تحديد البيانات المطلوب تسجيلها (أعراض، نتائج، التزام) |

**الأعمدة الرئيسية:**
- ProgramID (int, FK→PSPPrograms)
- DataEntryTitle (nvarchar(200))
- DataEntryType (int) - 1=نص, 2=رقم, 3=قائمة, 4=نعم/لا, 5=تاريخ, 6=ملف
- AllowedValues (nvarchar(1000)) - JSON array للقائمة
- RequiredFrom (int) - 1=مريض, 2=طبيب, 3=كلاهما
- RequiredTiming (int) - 1=قبل, 2=أثناء, 3=بعد, 4=دوري, 5=عند حدث

---

### جدول 14: `PSPTestResults` - نتائج التحاليل المسجلة

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | TestResultID (int) |
| **الغرض** | تسجيل نتائج التحاليل الفعلية |

**الأعمدة الرئيسية:**
- RequiredTestID (int, FK→PSPRequiredTests)
- PatientID (int, FK→PSPPatients)
- PerformedByUserID (nvarchar(450), FK→AspNetUsers)
- PerformedDate (datetime)
- ResultValue (decimal(18,2)) - للنتائج الرقمية
- ResultText (nvarchar(500)) - للنتائج النصية
- IsAbnormal (bit) - هل النتيجة غير طبيعية؟

---

### جدول 15: `PSPFollowUpRecords` - سجلات المتابعة المسجلة

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | FollowUpRecordID (int) |
| **الغرض** | تسجيل سجلات المتابعة الفعلية |

**الأعمدة الرئيسية:**
- RequiredFollowUpID (int, FK→PSPRequiredFollowUps)
- PatientID (int, FK→PSPPatients)
- PerformedByUserID (nvarchar(450), FK→AspNetUsers)
- PerformedDate (datetime)
- Summary (nvarchar(1000)) - ملخص المتابعة
- Recommendations (nvarchar(1000)) - التوصيات

---

### جدول 16: `PSPAdverseEvents` - الآثار الجانبية المسجلة

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | AdverseEventID (int) |
| **الغرض** | تسجيل الآثار الجانبية المبلغ عنها |

**الأعمدة الرئيسية:**
- PatientID (int, FK→PSPPatients)
- ProgramID (int, FK→PSPPrograms)
- MedicationID (int, FK→Medications) - الدواء المشتبه به
- EventName (nvarchar(200)) - اسم العرض
- Severity (int) - 1=خفيف, 2=متوسط, 3=شديد, 4=خطير
- IsSerious (bit) - هل هو خطير؟
- ActionTaken (int) - الإجراء المتخذ
- Outcome (int) - النتيجة

---

### جدول 17: `DoctorNotes` - ملاحظات الطبيب

| الخاصية | القيمة |
|---------|--------|
| **المفتاح الأساسي** | DoctorNoteID (int) |
| **الغرض** | تسجيل ملاحظات الطبيب السريرية |

**الأعمدة الرئيسية:**
- PatientID (int, FK→PSPPatients)
- ProgramID (int, FK→PSPPrograms)
- DoctorID (nvarchar(450), FK→AspNetUsers)
- NoteTitle (nvarchar(200))
- NoteContent (nvarchar(2000))
- NoteType (int) - 1=تقييم, 2=توصية, 3=متابعة, 4=استشارة
- IsSharedWithPatient (bit)

---

## القسم الثالث: العلاقات الكاملة


PSPPrograms (1) ───┬──→ (M) PSPRequiredTests ──→ (M) PSPTestResults
│
├──→ (M) PSPRequiredFollowUps ──→ (M) PSPFollowUpRecords
│
├──→ (M) PSPRequiredDataEntries
│
├──→ (M) PSPAdverseEvents
│
├──→ (M) DoctorNotes
│
└──→ (M) PSPParticipations ──→ (M) PSPPatients ──┬──→ (M) PSPeRXs ──→ (M) PSPDispensations
│
├──→ (M) PSPTestResults
│
├──→ (M) PSPFollowUpRecords
│
├──→ (M) PSPAdverseEvents
│
└──→ (M) DoctorNotes


---

## القسم الرابع: ملخص العدادات

| القسم | عدد الجداول |
|-------|--------------|
| الجداول القديمة | 10 |
| الجداول الجديدة | 7 |
| **الإجمالي** | **17** |


### 🤔 رأيي في الجداول الجديدة وإمكانية الدمج

تحليلي للجداول الـ 7 الجديدة:
الجدول	النوع	يمكن دمجه مع؟	رأيي
PSPRequiredTests	تكوين (Configuration)	❌ لا	يبقى منفصل - منطق مختلف
PSPRequiredFollowUps	تكوين (Configuration)	❌ لا	يبقى منفصل - منطق مختلف
PSPRequiredDataEntries	تكوين (Configuration)	❌ لا	يبقى منفصل - منطق مختلف
PSPTestResults	تنفيذ (Execution)	✅ مع PSPRequiredTests؟	لا - تكوين vs تنفيذ
PSPFollowUpRecords	تنفيذ (Execution)	✅ مع PSPRequiredFollowUps؟	لا - تكوين vs تنفيذ
PSPAdverseEvents	تنفيذ (Execution)	❌ لا	يبقى منفصل - مستقل بذاته
DoctorNotes	تنفيذ (Execution)	✅ مع PSPFollowUpRecords؟	ممكن جزئياً
اقتراحي للدمج (إن أردت تقليل عدد الجداول):
الخيار 1: دمج DoctorNotes مع PSPFollowUpRecords
لماذا؟

كلاهما تسجيلات من الطبيب

كلاهما مرتبط بمريض وبرنامج

يمكن جعل الملاحظات حقلاً إضافياً في متابعة

الهيكل المقترح بعد الدمج:

markdown
### PSPFollowUpRecords (مدمج)
- الأعمدة الأصلية كلها موجودة
- + Notes (nvarchar(2000)) - ملاحظة أساسية
- + DoctorNotesJSON (nvarchar(max)) - للملاحظات المتعددة إن احتاجها
توفير: جدول واحد (من 17 إلى 16 جدولاً)

الخيار 2: توحيد الجداول الثلاثة للتكوين (Configuration)
دمج PSPRequiredTests + PSPRequiredFollowUps + PSPRequiredDataEntries في جدول واحد:

الجدول الموحد: PSPRequiredItems

العمود	الوصف
RequiredItemID	PK
ProgramID	FK
ItemType	1=تحليل, 2=متابعة, 3=بيانات
Title	اسم العنصر
Description	وصف
RequiredFrom	1=مريض, 2=طبيب, 3=كلاهما
RequiredTiming	1=قبل, 2=أثناء, 3=بعد, 4=دوري
FrequencyDays	للتكرار
ConfigJSON	nvarchar(max) - للتخزين المرن
محتوى ConfigJSON حسب ItemType:

تحليل: { "NormalRangeMin": 0, "NormalRangeMax": 100, "Unit": "mg/dL" }

متابعة: { "FollowUpType": 1, "DurationMinutes": 30, "ReminderDaysBefore": 3 }

بيانات: { "DataEntryType": 3, "AllowedValues": ["خيار1","خيار2"] }

توفير: جدولان (من 17 إلى 15 جدولاً)

الخيار 3: توحيد الجداول الثلاثة للتنفيذ (Execution)
دمج PSPTestResults + PSPFollowUpRecords في جدول واحد:

الجدول الموحد: PSPExecutionRecords

العمود	الوصف
ExecutionRecordID	PK
PatientID	FK
PerformedByUserID	FK
RecordType	1=نتيجة تحليل, 2=سجل متابعة
RequiredItemID	FK (لجدول PSPRequiredItems الموحد)
PerformedDate	datetime
ResultJSON	nvarchar(max)
توفير: جدول واحد (من 17 إلى 14 جدولاً)

📊 مقارنة الخيارات:
الخيار	عدد الجداول	المرونة	سهولة الاستعلام	تعقيد التطوير
الوضع الحالي	17 جدولاً	عالية جداً	سهلة	متوسط
الخيار 1 (دمج DoctorNotes)	16 جدولاً	عالية	سهلة	منخفض
الخيار 2 (توحيد التكوين)	15 جدولاً	متوسطة	متوسطة	متوسط
الخيار 3 (توحيد التنفيذ)	14 جدولاً	متوسطة	متوسطة	مرتفع
💭 رأيي النهائي (الأهم)
هل الوضع الحالي صحيح؟
نعم، الوضع الحالي صحيح من الناحية المعمارية.

الأسباب:

فصل التكوين عن التنفيذ - مبدأ أساسي في قواعد البيانات

كل جدول له غرض محدد - يسهل الصيانة والتطوير

مرونة عالية - يمكن إضافة أنواع جديدة بدون تعديل هيكل الجداول

استعلامات بسيطة - كل جدول لوحده، بدون الحاجة لـ JSON parsing

هل تحتاج للدمج؟
لا، لا أنصح بالدمج في حالتك.

الأسباب:

المشروع لا يزال في بدايته - 17 جدولاً عدد مقبول جداً لنظام PSP متكامل

الدمج قد يسبب تعقيدات لاحقة - صيانة JSON صعبة، استعلامات أبطأ

الوضوح أهم من تقليل عدد الجداول - فريقك سيفهم النظام بسهولة أكبر

يمكنك إضافة الجداول تدريجياً - ابدأ بالأساسي، زود لاحقاً